In [ ]:
import os
import re
import time
import requests
import pandas as pd
from pathlib import Path

# ==========================================
# CONFIGURATION
# ==========================================
GITHUB_TOKEN = "temporary_token"  # Paste your token here
DATASET_DIR = "../yourrawdatasetfolder"      # Input folder containing your raw .txt files
OUTPUT_DIR = "../youroutputfolder"            # Folder where tiered CSVs will be saved

HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}


# ==========================================
# 1. PARSE AST BLOCK TEXT FILES
# ==========================================
def parse_ast_txt_file(file_path):
    records = []
    current_class, current_algo, current_lang = "UNKNOWN", "UNKNOWN", "UNKNOWN"
    current_repo, current_file, current_url, current_line_num, current_func = None, None, None, None, None

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            stripped = line.strip()
            if not stripped:
                continue

            if "Class:" in stripped and "Algo:" in stripped:
                header_match = re.search(r"Class:\s*(.*?)\s*\|\s*Algo:\s*(.*?)\s*\|\s*Lang:\s*(.*)", stripped)
                if header_match:
                    current_class = header_match.group(1).strip()
                    current_algo = header_match.group(2).strip()
                    current_lang = header_match.group(3).strip()
                continue

            if stripped.startswith("Repository:"):
                current_repo = stripped.replace("Repository:", "").strip().lower()
                current_file, current_url = None, None
                continue

            if stripped.startswith("File:"):
                current_file = stripped.replace("File:", "").strip()
                continue

            if stripped.startswith("URL:"):
                current_url = stripped.replace("URL:", "").strip()
                continue

            if stripped.startswith("-> Line"):
                line_match = re.search(r"-> Line\s+(\d+):\s*(.*)", stripped)
                if line_match:
                    current_line_num = line_match.group(1)
                    current_func = line_match.group(2).strip()
                continue

            if stripped.startswith("-> Parameters:"):
                params = stripped.replace("-> Parameters:", "").strip()
                if current_repo and current_file:
                    records.append({
                        "repo_name": current_repo,
                        "file_path": current_file,
                        "line_no": current_line_num,
                        "function": current_func,
                        "parameters": params,
                        "algorithm": current_algo,
                        "language": current_lang,
                        "class_type": current_class,
                        "url": current_url,
                        "source_file": str(file_path)
                    })
                continue

    return records


# ==========================================
# 2. FETCH GITHUB REPO STARS
# ==========================================
def get_repo_stars(repo_name, cache):
    if repo_name in cache:
        return cache[repo_name]
    
    clean_name = repo_name.replace("https://github.com/", "").replace("http://github.com/", "").replace(".git", "").strip("/")
    parts = clean_name.split("/")
    if len(parts) < 2:
        cache[repo_name] = 0
        return 0
    
    target_repo = f"{parts[0]}/{parts[1]}"
    url = f"https://api.github.com/repos/{target_repo}"
    
    try:
        res = requests.get(url, headers=HEADERS, timeout=10)
        if res.status_code == 200:
            stars = res.json().get("stargazers_count", 0)
            cache[repo_name] = stars
            return stars
        elif res.status_code in (403, 429):
            sleep_dur = max(int(res.headers.get("X-RateLimit-Reset", time.time() + 60)) - int(time.time()), 5)
            print(f"\n[Rate Limited] Pausing for {sleep_dur} seconds...")
            time.sleep(sleep_dur)
            return get_repo_stars(repo_name, cache)
    except Exception:
        pass
    
    cache[repo_name] = 0
    return 0


# ==========================================
# 3. RUN PIPELINE & EXPORT TIERS
# ==========================================
def run_full_pipeline():
    input_path = Path(DATASET_DIR)
    output_path = Path(OUTPUT_DIR)
    output_path.mkdir(parents=True, exist_ok=True)

    txt_files = list(input_path.glob("**/*.txt"))
    print(f"Found {len(txt_files)} block-formatted .txt files to parse...")

    all_records = []
    for file_path in txt_files:
        all_records.extend(parse_ast_txt_file(file_path))

    df = pd.DataFrame(all_records)
    if df.empty:
        print("No valid repository records found. Check your DATASET_DIR path.")
        return

    unique_repos = df["repo_name"].unique()
    print(f"Extracted {len(df)} total match rows across {len(unique_repos)} unique repositories.")
    print("Fetching star counts from GitHub API...")

    repo_cache = {}
    for idx, repo in enumerate(unique_repos, 1):
        get_repo_stars(repo, repo_cache)
        if idx % 20 == 0 or idx == len(unique_repos):
            print(f"  -> Queried {idx}/{len(unique_repos)} repositories...", end="\r")

    df["stars"] = df["repo_name"].map(repo_cache).fillna(0).astype(int)

   #  4-Tier Bins: Tier 1 (0-9), Tier 2 (10-49), Tier 3 (50-100), Tier 4 (101+)
    bins = [-1, 9, 49, 100, float("inf")]
    labels = ["Tier_1", "Tier_2", "Tier_3", "Tier_4"]
    df["star_tier"] = pd.cut(df["stars"], bins=bins, labels=labels)

    print("\n\n" + "="*50)
    print(" SAVING TIER FILES ")
    print("="*50)

    for tier_label in labels:
        tier_df = df[df["star_tier"] == tier_label]
        filename = f"{tier_label.lower()}.csv"
        dest = output_path / filename
        
        tier_df.to_csv(dest, index=False)
        unique_count = tier_df["repo_name"].nunique()
        print(f"[{tier_label}] Saved {len(tier_df)} rows ({unique_count} unique repos) to {filename}")

    print("\nPipeline execution complete!")
    return df

# Execute directly in your notebook cell:
df = run_full_pipeline()

Found 1 block-formatted .txt files to parse...
Extracted 3196 total match rows across 1194 unique repositories.
Fetching star counts from GitHub API...
  -> Queried 1194/1194 repositories...

 SAVING TIER FILES 
[Tier_1] Saved 2434 rows (962 unique repos) to tier_1.csv
[Tier_2] Saved 346 rows (122 unique repos) to tier_2.csv
[Tier_3] Saved 112 rows (29 unique repos) to tier_3.csv
[Tier_4] Saved 304 rows (81 unique repos) to tier_4.csv

Pipeline execution complete!
